# Unsupervised Domain Adaptation (DANN) for Plant Classification
Goal: Train a classifier that generalizes from single-plant photos to complex quadrat backgrounds using Domain-Adversarial Neural Networks.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import pandas as pd
import mlflow
import numpy as np
from dotenv import load_dotenv
import itertools

from src.models.dann import DANN
from src.data.datasets import DomainDataset, get_interleaved_loader
from src.utils.metrics import log_dann_metrics, AverageMeter

# 1. Load environment variables
load_dotenv()

REQUIRED_ENV_VARS = [
    "SOURCE_DATA_PATH",
    "TARGET_DATA_PATH",
    "BACKBONE",
    "LEARNING_RATE",
    "DOMAIN_LOSS_WEIGHT",
]


def validate_env():
    missing = [var for var in REQUIRED_ENV_VARS if os.getenv(var) is None]
    if missing:
        raise EnvironmentError(
            f"Missing required environment variables: {', '.join(missing)}"
        )
    print("Environment variables validated successfully.")


validate_env()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 1. Data Preparation

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# Load metadata (Paths defined in .env)
source_path = os.getenv("SOURCE_DATA_PATH")
target_path = os.getenv("TARGET_DATA_PATH")

# Placeholder: In a real scenario, these would be loaded from CSVs
source_df = pd.DataFrame({"image_path": [], "species_id": []})
target_df = pd.DataFrame({"image_path": []})

source_ds = DomainDataset(source_path, source_df, domain_label=0, transform=transform)
target_ds = DomainDataset(
    target_path,
    target_df,
    domain_label=1,
    transform=transform,
    is_source=False,
)

batch_size = int(os.getenv("BATCH_SIZE", 32))
source_loader = DataLoader(source_ds, batch_size=batch_size, shuffle=True)
target_loader = DataLoader(target_ds, batch_size=batch_size, shuffle=True)

## 2. Model Initialization

In [ ]:
backbone = os.getenv("BACKBONE")
num_classes = 1000  # Example number of species
model = DANN(backbone, num_classes).to(DEVICE)

lr = float(os.getenv("LEARNING_RATE"))
domain_loss_weight = float(os.getenv("DOMAIN_LOSS_WEIGHT", 1.0))

optimizer = optim.Adam(model.parameters(), lr=lr)
criterion_class = nn.CrossEntropyLoss()
criterion_domain = nn.BCELoss()

## 3. Training Loop with Adversarial Step

In [ ]:
def train_epoch(
    model, source_loader, target_loader, optimizer, epoch, num_epochs, domain_weight
):
    model.train()
    metrics = {
        "loss_class": AverageMeter(),
        "loss_domain_s": AverageMeter(),
        "loss_domain_t": AverageMeter(),
    }

    n_batches = len(source_loader)
    for i, (source_batch, target_batch) in enumerate(
        zip(source_loader, itertools.cycle(target_loader))
    ):
        # 1. Schedule alpha (Gradient Reversal Scaling)
        p = float(i + epoch * n_batches) / (num_epochs * n_batches)
        alpha = 2.0 / (1.0 + np.exp(-10 * p)) - 1

        # 2. Source domain pass
        s_img, s_cls_label, s_dom_label = source_batch
        s_img, s_cls_label = s_img.to(DEVICE), s_cls_label.to(DEVICE)
        s_dom_label = s_dom_label.float().to(DEVICE).view(-1, 1)

        optimizer.zero_grad()
        s_cls_out, s_dom_out = model(s_img, alpha)

        loss_s_cls = criterion_class(s_cls_out, s_cls_label)
        loss_s_dom = criterion_domain(s_dom_out, s_dom_label)

        # 3. Target domain pass
        t_img, _, t_dom_label = target_batch
        t_img = t_img.to(DEVICE)
        t_dom_label = t_dom_label.float().to(DEVICE).view(-1, 1)

        _, t_dom_out = model(t_img, alpha)
        loss_t_dom = criterion_domain(t_dom_out, t_dom_label)

        # 4. Total loss and update (apply domain_loss_weight)
        loss = loss_s_cls + domain_weight * (loss_s_dom + loss_t_dom)
        loss.backward()
        optimizer.step()

        metrics["loss_class"].update(loss_s_cls.item())
        metrics["loss_domain_s"].update(loss_s_dom.item())
        metrics["loss_domain_t"].update(loss_t_dom.item())

    return {k: v.avg for k, v in metrics.items()}

## 4. Main Execution

In [ ]:
num_epochs = int(os.getenv("NUM_EPOCHS", 20))
with mlflow.start_run() as run:
    mlflow.log_params(
        {
            "backbone": backbone,
            "learning_rate": lr,
            "batch_size": batch_size,
            "domain_loss_weight": domain_loss_weight,
        }
    )

    for epoch in range(num_epochs):
        epoch_metrics = train_epoch(
            model,
            source_loader,
            target_loader,
            optimizer,
            epoch,
            num_epochs,
            domain_loss_weight,
        )
        log_dann_metrics(epoch_metrics, step=epoch)
        print(f"Epoch {epoch}: {epoch_metrics}")